# Proyecto Oráculo — Notebook de evaluación

**Autor:** Juan Felipe Henao · **Grupo:** ADAG1 · Análisis y Diseño de Algoritmos 2026-B

Este notebook carga la configuración de `entrega.json` y la evalúa sobre un conjunto de instancias. **No requiere editar código, rutas ni parámetros.**

**Cómo ejecutarlo:**

1. **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU.**
2. Ejecutar la celda 1 y, cuando termine, **Entorno de ejecución ▸ Reiniciar sesión** (lo exige la librería del modelo).
3. Ejecutar el resto con **Entorno de ejecución ▸ Ejecutar todas** (la celda 1 se puede volver a ejecutar sin problema).

**Datos de prueba.** El notebook busca automáticamente un archivo `.json` de instancias, con el mismo formato de `datos_visibles.json` (lista de instancias con `prompt`, `ids` y `kwargs`), en la carpeta `/content` de Colab. Para usar los datos privados basta con arrastrar el archivo al panel de archivos de Colab, con cualquier nombre, antes de la celda 4. Si no encuentra ninguno, ofrece subirlo; si no se sube, evalúa sobre la partición de validación pública como demostración.

## 1 · Instalar y bajar los archivos del curso
*Al terminar, reinicia la sesión y sigue en la celda 2.*

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
![ -d open-instruct ] || git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-Oraculo-AyD/main"
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

import io, urllib.request, zipfile
NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
for carpeta, nombre in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"),
                        ("taggers", "averaged_perceptron_tagger"),
                        ("taggers", "averaged_perceptron_tagger_eng")]:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")

## 2 · Cargar y validar `entrega.json`

Si `entrega.json` está en `/content`, se usa esa copia. Si no, se descarga del repositorio del proyecto. Se valida que cada ranura tenga un índice entre 0 y 7 y que la temperatura sea 0.0, 0.3 o 0.7.

In [ ]:
import json, os, urllib.request

URL_ENTREGA = "https://raw.githubusercontent.com/Pipe-1z/proyecto-oraculo-adag1/main/entrega.json"
RANURAS = ["rol", "estrategia", "formato", "estilo", "cierre"]

if os.path.exists("entrega.json"):
    entrega = json.load(open("entrega.json", encoding="utf-8"))
    origen = "archivo local entrega.json"
else:
    with urllib.request.urlopen(URL_ENTREGA) as resp:
        entrega = json.loads(resp.read().decode("utf-8"))
    origen = URL_ENTREGA

config_entrega = entrega["config"]
for r in RANURAS:
    assert isinstance(config_entrega[r], int) and 0 <= config_entrega[r] <= 7, f"{r} inválido: {config_entrega[r]}"
assert float(config_entrega["temperatura"]) in (0.0, 0.3, 0.7), "temperatura inválida"

print("entrega.json cargado desde:", origen)
print(json.dumps(entrega, indent=1, ensure_ascii=False))

## 3 · Modelo y oráculo
Se usa el mismo modelo con el que se hizo la búsqueda (`qwen17b`) y un caché propio, para no mezclar respuestas con las de la búsqueda.

In [ ]:
from ayudas import cargar_modelo, cargar_datos, dividir
from oraculo import Oraculo, espacio
try:
    from oraculo import TEMPERATURAS
except ImportError:
    TEMPERATURAS = [0.0, 0.3, 0.7]

MODELO = "qwen17b"
SEMILLA = 1

modelo = cargar_modelo(MODELO)
datos = cargar_datos("datos_visibles.json")
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion, cache="cache_evaluacion.json", lote=16, presupuesto=32_000_000)

# La configuración se toma del espacio oficial para garantizar que sea válida.
INDICE = {tuple(c[r] for r in RANURAS) + (float(c.get("temperatura", 0.0)),): c
          for c in espacio(TEMPERATURAS)}
config = INDICE[tuple(config_entrega[r] for r in RANURAS) + (float(config_entrega["temperatura"]),)]
print("Configuración a evaluar:", config)

## 4 · Localizar los datos de prueba

Orden de búsqueda:

1. Variable de entorno `DATOS_PRIVADOS` con la ruta del archivo, si existe.
2. Cualquier `.json` en `/content` (o en la carpeta actual) con formato de instancias, distinto de los archivos del curso y del proyecto.
3. Si no hay ninguno, se ofrece subirlo desde el computador.
4. Si no se sube nada, se usa la partición de validación pública (87 instancias) como demostración.

In [ ]:
import glob

EXCLUIR = {"datos_visibles.json", "entrega.json", "cache_evaluacion.json",
           "cache_oraculo.json", "resultado_evaluacion.json"}

def es_archivo_de_instancias(ruta):
    try:
        d = json.load(open(ruta, encoding="utf-8"))
    except Exception:
        return False
    return isinstance(d, list) and len(d) > 0 and isinstance(d[0], dict) \
        and "prompt" in d[0] and "ids" in d[0]

def cargar_instancias(ruta):
    d = json.load(open(ruta, encoding="utf-8"))
    utiles = [x for x in d if x.get("split") != "descartada"]
    return utiles or d

ruta_privada = os.environ.get("DATOS_PRIVADOS")
if not (ruta_privada and es_archivo_de_instancias(ruta_privada)):
    ruta_privada = None
    carpetas = {"/content", os.getcwd()}
    candidatos = sorted({p for c in carpetas for p in glob.glob(os.path.join(c, "*.json"))})
    candidatos = [p for p in candidatos if os.path.basename(p) not in EXCLUIR and es_archivo_de_instancias(p)]
    if candidatos:
        ruta_privada = candidatos[0]

if ruta_privada is None:
    try:
        from google.colab import files
        print("No se encontró un archivo de instancias. Puede subirlo ahora (o cancelar para usar validación):")
        subidos = files.upload()
        for nombre in subidos:
            if es_archivo_de_instancias(nombre):
                ruta_privada = nombre
                break
    except Exception as e:
        print("Sin subida de archivos:", e)

if ruta_privada:
    INSTANCIAS = cargar_instancias(ruta_privada)
    FUENTE = f"datos de prueba: {ruta_privada}"
else:
    INSTANCIAS = None
    FUENTE = "partición de validación pública (demostración)"
print("Se evaluará sobre:", FUENTE)

## 5 · Evaluación

In [ ]:
from collections import Counter

if INSTANCIAS is not None:
    r = oraculo.evaluar(config, INSTANCIAS, semilla=SEMILLA)
    n = len(INSTANCIAS)
else:
    r = oraculo.validar(config, n=None)
    n = len(validacion)

fallos = Counter(str(t["violo"]) for t in r.trazas if t.get("violo"))
familias = Counter(k.split(":")[0] for k in fallos.elements())

print("Fuente:       ", FUENTE)
print("Instancias:   ", n)
print("Configuración:", {k: config[k] for k in RANURAS}, "temperatura", config.get("temperatura", 0.0))
print(f"PRECISIÓN:     {r.precision:.1%}")
print("\nFallos por familia de restricción:")
for f, c in familias.most_common():
    print(f"  {f:22s} {c}")

## 6 · Guardar el resultado
Se escribe `resultado_evaluacion.json` con la fuente, el número de instancias, la configuración y la precisión.

In [ ]:
resultado = {
    "grupo": entrega.get("grupo"),
    "semana": entrega.get("semana"),
    "config": entrega["config"],
    "fuente": FUENTE,
    "instancias": n,
    "precision": r.precision,
    "fallos_por_familia": dict(familias.most_common()),
}
json.dump(resultado, open("resultado_evaluacion.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)
print(json.dumps(resultado, indent=1, ensure_ascii=False))